In [ ]:
from google.cloud import aiplatform

PROJECT_ID = "your_project"
REGION = "asia-southeast1"

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    experiment="churn-model-comparison"
)

In [ ]:
from kfp.v2.dsl import component

@component(
    packages_to_install=[
        "pandas",
        "google-cloud-bigquery",
        "scikit-learn"
    ]
)
def load_data(output_path: str):

    from google.cloud import bigquery
    import pandas as pd
    from sklearn.model_selection import train_test_split

    client = bigquery.Client()

    query = """
    SELECT * EXCEPT(feature_timestamp, customer_id)
    FROM `your_project.your_dataset.customer_churn_demo`
    """

    df = client.query(query).to_dataframe()

    # simple encoding
    df = pd.get_dummies(df)

    X = df.drop("churned", axis=1)
    y = df["churned"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    X_train.to_csv(output_path + "_X_train.csv", index=False)
    X_test.to_csv(output_path + "_X_test.csv", index=False)
    y_train.to_csv(output_path + "_y_train.csv", index=False)
    y_test.to_csv(output_path + "_y_test.csv", index=False)

In [ ]:
@component(
    packages_to_install=["pandas","scikit-learn","google-cloud-aiplatform"]
)
def train_rf(data_path: str):

    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score
    from google.cloud import aiplatform

    aiplatform.start_run("random-forest")

    X_train = pd.read_csv(data_path + "_X_train.csv")
    y_train = pd.read_csv(data_path + "_y_train.csv")
    X_test = pd.read_csv(data_path + "_X_test.csv")
    y_test = pd.read_csv(data_path + "_y_test.csv")

    model = RandomForestClassifier(n_estimators=50)
    model.fit(X_train, y_train.values.ravel())

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    aiplatform.log_metrics({"accuracy": acc})
    aiplatform.end_run()

In [ ]:
@component(
    packages_to_install=["pandas","xgboost","google-cloud-aiplatform"]
)
def train_xgb(data_path: str):

    import pandas as pd
    import xgboost as xgb
    from sklearn.metrics import accuracy_score
    from google.cloud import aiplatform

    aiplatform.start_run("xgboost")

    X_train = pd.read_csv(data_path + "_X_train.csv")
    y_train = pd.read_csv(data_path + "_y_train.csv")
    X_test = pd.read_csv(data_path + "_X_test.csv")
    y_test = pd.read_csv(data_path + "_y_test.csv")

    model = xgb.XGBClassifier()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    aiplatform.log_metrics({"accuracy": acc})
    aiplatform.end_run()

In [ ]:
@component(
    packages_to_install=["pandas","tensorflow","google-cloud-aiplatform"]
)
def train_nn(data_path: str):

    import pandas as pd
    import tensorflow as tf
    from sklearn.metrics import accuracy_score
    from google.cloud import aiplatform

    aiplatform.start_run("neural-network")

    X_train = pd.read_csv(data_path + "_X_train.csv")
    y_train = pd.read_csv(data_path + "_y_train.csv")
    X_test = pd.read_csv(data_path + "_X_test.csv")
    y_test = pd.read_csv(data_path + "_y_test.csv")

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(X_train, y_train, epochs=10, verbose=0)

    preds = (model.predict(X_test) > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)

    aiplatform.log_metrics({"accuracy": acc})
    aiplatform.end_run()

In [ ]:
from kfp.v2.dsl import pipeline

@pipeline(name="churn-parallel-training")
def churn_pipeline():

    data_task = load_data()

    rf = train_rf(data_task.output)
    xgb = train_xgb(data_task.output)
    nn = train_nn(data_task.output)

In [ ]:
from kfp.v2 import compiler

compiler.Compiler().compile(
    pipeline_func=churn_pipeline,
    package_path="churn_pipeline.json"
)

In [ ]:
job = aiplatform.PipelineJob(
    display_name="churn-training",
    template_path="churn_pipeline.json",
)

job.run()